# PySpark Delta Lake MERGE Pipeline

## Step 1: Initialize PySpark Session with Delta Extension

In [1]:
import delta
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, count
from delta.tables import DeltaTable

builder = SparkSession.builder \
    .appName('DeltaLakeMERGEAssignment') \
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension') \
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog') \
    .master('local[*]')

spark = delta.configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark Session Initialized Successfully!')

Spark Session Initialized Successfully!


## Step 2: Load Clean Master Data into Delta Table

In [2]:
df_clean = spark.read.option('header', 'true').option('inferSchema', 'false').csv('../data/customer_master.csv')
df_clean = df_clean.toDF('customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region')
df_clean = df_clean.withColumn('postal_code', col('postal_code').cast('string')).fillna({'postal_code': '00000', 'region': 'Unknown'})
df_clean = df_clean.dropDuplicates(['customer_id'])

scd1_path = '../delta_tables/customer_scd1'
df_clean.write.format('delta').mode('overwrite').save(scd1_path)
print('Clean Master Delta Table Count:', df_clean.count())

Clean Master Delta Table Count: 100


## Step 3: Load Incremental Data (data/customer_incremental.csv)

In [3]:
df_inc = spark.read.option('header', 'true').option('inferSchema', 'false').csv('../data/customer_incremental.csv')
df_inc = df_inc.toDF('customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region')
df_inc = df_inc.withColumn('postal_code', col('postal_code').cast('string')).fillna({'postal_code': '00000', 'region': 'Unknown'})

print('Incremental Dataset Count:', df_inc.count())
df_inc.show(15, truncate=False)

Incremental Dataset Count: 15
+-----------+--------------------+-----------+-------------+------------+-------------+-----------+-------+
|customer_id|customer_name       |segment    |country      |city        |state        |postal_code|region |
+-----------+--------------------+-----------+-------------+------------+-------------+-----------+-------+
|AA-10315   |Alex Avila          |Corporate  |United States|New York    |New York     |55407.0    |Central|
|AA-10375   |Allen Armold        |Home Office|United States|Los Angeles |California   |85204.0    |West   |
|AA-10480   |Andrew Allen        |Consumer   |United States|Chicago     |Illinois     |28027.0    |South  |
|AA-10645   |Anna Andreadi       |Corporate  |United States|Houston     |Texas        |19013.0    |East   |
|AB-10015   |Aaron Bergman       |Home Office|United States|Phoenix     |Arizona      |98103.0    |West   |
|AB-10060   |Adam Bellavance     |Consumer   |United States|Philadelphia|Pennsylvania |10009.0    |East   

## Step 4: Apply Delta MERGE Operation & Display Results

### Record Changes:
- **Updates (10 Records)**: `AA-10315`, `AA-10375`, `AA-10480`, `AA-10645`, etc.
- **Inserts (5 Records)**: `NEW-001` to `NEW-005`


In [4]:
delta_scd1 = DeltaTable.forPath(spark, scd1_path)

delta_scd1.alias('target').merge(
    df_inc.alias('source'),
    'target.customer_id = source.customer_id'
).whenMatchedUpdate(set={
    'customer_name': 'source.customer_name',
    'segment': 'source.segment',
    'country': 'source.country',
    'city': 'source.city',
    'state': 'source.state',
    'postal_code': 'source.postal_code',
    'region': 'source.region'
}).whenNotMatchedInsert(values={
    'customer_id': 'source.customer_id',
    'customer_name': 'source.customer_name',
    'segment': 'source.segment',
    'country': 'source.country',
    'city': 'source.city',
    'state': 'source.state',
    'postal_code': 'source.postal_code',
    'region': 'source.region'
}).execute()

df_scd1_final = spark.read.format('delta').load(scd1_path)
df_scd1_final.orderBy('customer_id').show(15, truncate=False)

+-----------+--------------------+-----------+-------------+-------------+------------+-----------+-------+
|customer_id|customer_name       |segment    |country      |city         |state       |postal_code|region |
+-----------+--------------------+-----------+-------------+-------------+------------+-----------+-------+
|AA-10315   |Alex Avila          |Corporate  |United States|New York     |New York    |55407.0    |Central|
|AA-10375   |Allen Armold        |Home Office|United States|Los Angeles  |California  |85204.0    |West   |
|AA-10480   |Andrew Allen        |Consumer   |United States|Chicago      |Illinois    |28027.0    |South  |
|AA-10645   |Anna Andreadi       |Corporate  |United States|Houston      |Texas       |19013.0    |East   |
|AB-10015   |Aaron Bergman       |Home Office|United States|Phoenix      |Arizona     |98103.0    |West   |
|AB-10060   |Adam Bellavance     |Consumer   |United States|Philadelphia |Pennsylvania|10009.0    |East   |
|AB-10105   |Adrian Barton  

## Step 5: Validation & Transaction History

In [5]:
scd1_count = df_scd1_final.count()
unique_keys = df_scd1_final.select('customer_id').distinct().count()
print(f'Total Rows: {scd1_count}, Unique Customer IDs: {unique_keys}')
assert scd1_count == unique_keys, 'Validation Failed: Duplicates found!'
print('Validation Passed: Zero duplicate customer IDs.')

delta_scd1.history().select('version', 'timestamp', 'operation', 'operationMetrics').show(truncate=False)

Total Rows: 105, Unique Customer IDs: 105
Validation Passed: Zero duplicate customer IDs.
+-------+-----------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation|operationMetrics                                                                                                                                         

## Step 6: Final Output & Delta Time Travel

In [6]:
v0_df = spark.read.format('delta').option('versionAsOf', 0).load(scd1_path)
v1_df = spark.read.format('delta').option('versionAsOf', 1).load(scd1_path)

print('Version 0 Row Count:', v0_df.count())
print('Version 1 Row Count:', v1_df.count())

v0_df.orderBy('customer_id').show(5, truncate=False)
v1_df.orderBy('customer_id').show(5, truncate=False)

Version 0 Row Count: 100
Version 1 Row Count: 100
+-----------+-------------+--------+-------------+-----------+--------------+-----------+-------+
|customer_id|customer_name|segment |country      |city       |state         |postal_code|region |
+-----------+-------------+--------+-------------+-----------+--------------+-----------+-------+
|AA-10315   |Alex Avila   |Consumer|United States|Minneapolis|Minnesota     |55407.0    |Central|
|AA-10375   |Allen Armold |Consumer|United States|Mesa       |Arizona       |85204.0    |West   |
|AA-10480   |Andrew Allen |Consumer|United States|Concord    |North Carolina|28027.0    |South  |
|AA-10645   |Anna Andreadi|Consumer|United States|Chester    |Pennsylvania  |19013.0    |East   |
|AB-10015   |Aaron Bergman|Consumer|United States|Seattle    |Washington    |98103.0    |West   |
+-----------+-------------+--------+-------------+-----------+--------------+-----------+-------+
only showing top 5 rows
+-----------+-------------+--------+--------